In [96]:
def pp(str):
    print(str)
    print('=' * 100)

1 Load it, and remove the empty columns
Load bike_rentals.csv and make a copy. Print the shape and the data types. Which type
does price have? Then find the columns where every value is missing, print their names,
and remove them. How many columns are left, and how many empty columns?

Answer: 12 left, 3 empty

In [97]:
import pandas as pd

rentals_raw = pd.read_csv('bike_rentals.csv')
rentals = rentals_raw.copy()
pp(rentals.shape)
pp(rentals.dtypes)

empty_columns = rentals.columns[rentals.isna().all()]
pp(empty_columns.tolist())
rentals = rentals.drop(columns=empty_columns)
pp(rentals.shape)

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object
['coupon_code', 'damage_notes', 'late_fee_note']
(403, 12)


2 The price: from text to numbers
Look at the first rows of price. It is text with a currency in front and a comma for
thousands. Remove the two things in two separate steps, and print the first three values
after each step. Then change the type to float, check the type, and run describe() on the
price. Finally, show the three most expensive rentals (rental_id, bike_type, status and
price).

In [98]:
# SEK 1,311.50

rentals['price'] = rentals['price'].str.replace('SEK ', '')
pp(rentals.head(3))
rentals['price'] = rentals['price'].str.replace(',', '')
pp(rentals.head(3))

pp(rentals['price'].dtype)
rentals['price'] = rentals['price'].astype(float)
pp(rentals['price'].dtype)
pp(rentals['price'].describe().round(0))

pp(rentals[['rental_id', 'bike_type', 'status', 'price']]
    .sort_values('price', ascending=False).head(3))

  rental_id    station  bike_type helmets_text is_member insured paid_online  \
0    R-5020  Södermalm     E-bike     1 helmet         f       t           t   
1    R-5100  Södermalm     E-bike     1 helmet         f       t           t   
2    R-5055  Södermalm  City bike    3 helmets         f       f           t   

      status  start_date    end_date     price  rating  
0  Completed  2026-04-27  2026-04-29    855.50     5.0  
1  Completed  2026-09-16  2026-09-19  1,311.50     4.0  
2  Completed  2026-08-04  2026-08-06    350.00     NaN  
  rental_id    station  bike_type helmets_text is_member insured paid_online  \
0    R-5020  Södermalm     E-bike     1 helmet         f       t           t   
1    R-5100  Södermalm     E-bike     1 helmet         f       t           t   
2    R-5055  Södermalm  City bike    3 helmets         f       f           t   

      status  start_date    end_date    price  rating  
0  Completed  2026-04-27  2026-04-29   855.50     5.0  
1  Completed  2026

3 The missing prices
How many prices are missing, and how many percent is that? Is there a reason? Compare
with the column status: use pd.crosstab on "price is missing" and "the rental was
cancelled". Which missing prices have a reason, and which are missing by accident? Write
what you decide to do with them, as a comment.

Answer:
47 missing prices for cancelled rentals
4 missing prices for completed rentals (by accident?) - ignore those

In [99]:
pp(rentals['price'].isna().sum())
pp((rentals['price'].isna().mean() * 100).round(2))
pp((rentals['status'] == 'Cancelled').sum())

missing_price = rentals['price'].isna()
pd.crosstab(missing_price, rentals['status'], 
    rownames=['Price is missing'], 
    colnames=['Rental status'])

# pp(rentals.loc[(missing_price) & (rentals['status'] == 'Completed')])

51
12.66
47


Rental status,Cancelled,Completed,No-show
Price is missing,,,
False,0,330,22
True,47,4,0


4 The letters t and f
Find the columns that contain only t and f (and maybe missing values), with a list
comprehension. Why is status not in the list? Change the columns to True and False with
map. How many missing values does each column have afterwards? Then compare the
median price of members and non-members, with the number of rentals behind each
number.

Answer:
status is not a boolean

In [100]:
tf_columns = [
    col for col in rentals.columns
    if rentals[col].dropna().isin(['t', 'f']).all()
]
pp(tf_columns)

for col in tf_columns:
    rentals[col] = rentals[col].map({'t': True, 'f': False})

# pp(rentals[['is_member', 'insured', 'paid_online']].head())

pp(rentals[['is_member', 'insured', 'paid_online']].isna().sum())

pp(rentals.groupby('is_member')['price'].agg(['count', 'median']))


['is_member', 'insured', 'paid_online']
is_member      0
insured        3
paid_online    0
dtype: int64
           count  median
is_member               
False        248  756.25
True         104  558.00
